# Module 8: Conditioning & Guidance

Conditioning is what transforms a diffusion model from a novelty into a **controllable** generative tool. Without it, the model generates random samples from the learned distribution. With it, we can say "generate a 7" or "generate a cat" — and the model obeys.

Here's what we'll work through together:

- **Class-conditional generation** — injecting a class label into the UNet
- **Classifier guidance** — steering sampling with a separate classifier's gradients
- **Classifier-free guidance (CFG)** — the dominant modern approach, used by Stable Diffusion, DALL-E, Imagen, and Midjourney

By the end, you'll have a complete CFG pipeline that generates specific MNIST digits on demand — and you'll understand exactly how every production diffusion system handles conditioning.

**Estimated time:** 3–4 hours

**Key references:**
- [Classifier-Free Diffusion Guidance](https://arxiv.org/abs/2207.12598) — Ho & Salimans 2022
- [Diffusion Models Beat GANs on Image Synthesis (ADM)](https://arxiv.org/abs/2105.05233) — Dhariwal & Nichol 2021
- [Imagen](https://arxiv.org/abs/2205.11487) — Saharia et al. 2022

In [ ]:
import sys
import math
from typing import Optional, Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, '.')
from utils.unet import UNet
from utils.diffusion import prepare_schedule, ddpm_sample_step
from utils.schedule import cosine_schedule, get_schedule
from utils.visualization import show_images, denormalize, plot_loss_curve, show_denoising_trajectory, set_style
from utils.data import get_mnist_dataloader, get_device

torch.manual_seed(42)
device = get_device()
print(f"Using device: {device}")
set_style()

In [ ]:
# Noise schedule setup — used throughout this notebook
T = 1000
schedule = prepare_schedule(cosine_schedule(T), device)
schedule_cpu = prepare_schedule(cosine_schedule(T), torch.device("cpu"))

# Convenience aliases for inline forward diffusion
sqrt_alphas_cumprod = schedule["sqrt_alphas_cumprod"]
sqrt_one_minus_alphas_cumprod = schedule["sqrt_one_minus_alphas_cumprod"]

print(f"Schedule ready: T={T}, device={device}")

---
## 8.1 — Class-Conditional Generation

The simplest form of conditioning: tell the model **which class** to generate by feeding in a class label alongside the noisy image and timestep. Let's see how it works.

### Embedding the class label

1. **Embed the class label** using `nn.Embedding(num_classes + 1, embed_dim)` — the `+1` reserves an extra index as the "null token" (we'll need this later for CFG)
2. **Add the label embedding to the timestep embedding** — both have shape `(B, embed_dim)`, so they combine via element-wise addition
3. **Inject into each residual block** — the combined embedding passes through the same MLP that already processes the timestep, so it modulates every layer of the UNet

### How the embedding reaches the feature maps

The label embedding has shape `(B, embed_dim)`. After the MLP projects it, the result is reshaped to `(B, channels, 1, 1)` and **added to the feature maps** in each residual block. This is how a simple integer label influences every spatial location in the network.

> **Intuition:** The timestep embedding already tells the network "how noisy is this image?" Adding the class embedding extends this to "how noisy is this image, and what class should it be?"

### Architecture overview

Our `UNet` from `utils.unet` already supports class conditioning — we just need to pass `num_classes` when constructing it. The key changes from the unconditional version are small but important:

- **`nn.Embedding(num_classes + 1, time_embed_dim)`** maps each class label (0–9 for MNIST, plus a null token at index 10) to a learned vector
- The class embedding is **added** to the timestep embedding before being passed into each residual block
- When `class_label=None` is passed, the model behaves unconditionally (no class embedding added)

Everything else — the encoder/decoder structure, skip connections, attention blocks, sinusoidal timestep embedding — remains identical to what we built in Module 4.

In [ ]:
# Quick sanity check: verify shapes with and without class conditioning
torch.manual_seed(42)

# Unconditional
model_uncond = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4), num_classes=None)
x_test = torch.randn(2, 1, 28, 28)
t_test = torch.randint(0, 1000, (2,))
out_uncond = model_uncond(x_test, t_test)
print(f"Unconditional output shape: {out_uncond.shape}")  # (2, 1, 28, 28)

# Class-conditional (10 MNIST classes)
model_cond = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4), num_classes=10)
labels_test = torch.tensor([3, 7])
out_cond = model_cond(x_test, t_test, class_label=labels_test)
print(f"Conditional output shape:   {out_cond.shape}")    # (2, 1, 28, 28)

# Null token (unconditional pass through conditional model)
null_labels = torch.full((2,), 10)  # index 10 = null token for 10-class model
out_null = model_cond(x_test, t_test, class_label=null_labels)
print(f"Null-token output shape:    {out_null.shape}")    # (2, 1, 28, 28)

total_params = sum(p.numel() for p in model_cond.parameters())
print(f"\nConditional model parameters: {total_params:,}")

del model_uncond, model_cond  # free memory

### Exercise 8.1: Class-conditional training

Now it's your turn. Train a class-conditional diffusion model on MNIST. The training loop is nearly identical to the unconditional version from Module 6 — the only difference is passing `class_label=labels` to the model.

- Use `get_mnist_dataloader(batch_size=64)` for data
- Use `cosine_schedule(T=1000)` for the noise schedule
- Build a `UNet(image_channels=1, base_channels=64, channel_mults=(1,2,4), num_classes=10)`
- Train for 3000 steps with Adam (lr=2e-4)
- Pass labels directly: `model(x_t, t, class_label=labels)`

In [ ]:
# YOUR CODE HERE — Exercise 8.1

torch.manual_seed(42)

# Setup
model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4), num_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
dataloader = get_mnist_dataloader(batch_size=64)
data_iter = iter(dataloader)
losses = []
num_train_steps = 3000

model.train()
for step in tqdm(range(num_train_steps), desc="Training class-conditional model"):
    try:
        images, labels = next(data_iter)
    except StopIteration:
        data_iter = iter(dataloader)
        images, labels = next(data_iter)

    images, labels = images.to(device), labels.to(device)
    B = images.shape[0]

    # ===================== YOUR CODE HERE =====================
    # 1. Sample random timesteps t ~ Uniform(0, T)
    # 2. Sample noise epsilon ~ N(0, I)
    # 3. Create noisy images x_t using sqrt_alphas_cumprod and sqrt_one_minus_alphas_cumprod
    # 4. Predict noise: noise_pred = model(x_t, t, class_label=labels)
    # 5. Compute MSE loss between noise_pred and noise
    # 6. Backprop and step
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================

    # losses.append(loss.item())  # uncomment after implementing

# Tests — run this cell to check your work
assert len(losses) == num_train_steps, f"Expected {num_train_steps} loss entries but got {len(losses)} — did you append loss.item() each step?"
assert losses[-1] < 0.5, f"Final loss is {losses[-1]:.4f} — should be below 0.5 after {num_train_steps} steps"
print(f"Final loss: {losses[-1]:.4f}")
plot_loss_curve(losses, title="Exercise 8.1: Class-Conditional Training Loss")
print("Class-conditional training ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

torch.manual_seed(42)

model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4), num_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
dataloader = get_mnist_dataloader(batch_size=64)
data_iter = iter(dataloader)
losses = []
num_train_steps = 3000

model.train()
for step in tqdm(range(num_train_steps), desc="Training class-conditional model"):
    try:
        images, labels = next(data_iter)
    except StopIteration:
        data_iter = iter(dataloader)
        images, labels = next(data_iter)

    images, labels = images.to(device), labels.to(device)
    B = images.shape[0]

    t = torch.randint(0, T, (B,), device=device)
    noise = torch.randn_like(images)
    x_t = (
        sqrt_alphas_cumprod[t, None, None, None] * images
        + sqrt_one_minus_alphas_cumprod[t, None, None, None] * noise
    )

    noise_pred = model(x_t, t, class_label=labels)  # (B, 1, 28, 28)
    loss = F.mse_loss(noise_pred, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Final loss: {losses[-1]:.4f}")
plot_loss_curve(losses, title="Exercise 8.1: Class-Conditional Training Loss")
print("Class-conditional training ✓")

---
## 8.2 — Classifier Guidance

Before CFG existed, [Dhariwal & Nichol (2021)](https://arxiv.org/abs/2105.05233) proposed **classifier guidance**: use a separately trained classifier $p_\phi(y \mid x_t)$ to steer the diffusion sampling process toward a desired class $y$. Let's understand how it works — and why it was eventually replaced.

### The algorithm

At each sampling step $t$:

1. Predict noise with the (unconditional) diffusion model: $\hat{\epsilon} = \epsilon_\theta(x_t, t)$
2. Compute the classifier gradient: $g = \nabla_{x_t} \log p_\phi(y \mid x_t)$
3. Shift the noise prediction: $\tilde{\epsilon} = \hat{\epsilon} - s \cdot \sqrt{1 - \bar{\alpha}_t} \cdot g$
4. Use $\tilde{\epsilon}$ in the standard DDPM update step

Here $s$ is the **guidance scale** — higher $s$ produces sharper, more class-consistent images at the cost of diversity.

### The problem

This requires training a **separate classifier on noisy images** (not clean images — the classifier must handle arbitrary noise levels). This is wasteful and inflexible, which motivates classifier-free guidance in Section 8.3.

### Worked example: Noisy-image classifier

Let's train a small CNN classifier on **noisy** MNIST images at various noise levels. This classifier must handle noisy inputs because during sampling, $x_t$ is noisy — a clean-image classifier would fail completely.

The classifier takes two inputs:
- The noisy image `x_t` with shape `(B, 1, 28, 28)`
- The timestep `t` with shape `(B,)`, so it knows how much noise is present

In [ ]:
from utils.unet import SinusoidalTimestepEmbedding


class NoisyClassifier(nn.Module):
    """A simple CNN classifier for noisy MNIST images.

    Takes a noisy image and a timestep as input, and predicts class logits.
    The timestep embedding lets the classifier account for the noise level --
    a highly noisy image needs different features than a clean one.

    Architecture:
        - Sinusoidal timestep embedding projected to match channel dimensions
        - 3 conv layers with ReLU and downsampling
        - Global average pooling → FC → logits
    """

    def __init__(self, num_classes: int = 10, time_embed_dim: int = 64) -> None:
        super().__init__()
        self.time_embed = SinusoidalTimestepEmbedding(time_embed_dim)
        self.time_proj = nn.Linear(time_embed_dim, 1 * 28 * 28)  # project to image size

        # Conv layers: 1 → 32 → 64 → 128
        self.conv1 = nn.Conv2d(2, 32, 3, padding=1)    # 2 input channels: image + time map
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2)                     # halves spatial dims each time

        # After 3 poolings: 28 → 14 → 7 → 3
        self.fc1 = nn.Linear(128 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Classify a noisy image given its noise level.

        Args:
            x: (B, 1, 28, 28) noisy image.
            t: (B,) integer timestep indicating noise level.

        Returns:
            (B, num_classes) logits.
        """
        B = x.shape[0]

        # Create a timestep feature map and concatenate with the image
        t_emb = self.time_embed(t)                      # (B, time_embed_dim)
        t_map = self.time_proj(t_emb)                   # (B, 1*28*28)
        t_map = t_map.view(B, 1, 28, 28)               # (B, 1, 28, 28)
        h = torch.cat([x, t_map], dim=1)                # (B, 2, 28, 28)

        # Conv backbone
        h = self.pool(F.relu(self.conv1(h)))            # (B, 32, 14, 14)
        h = self.pool(F.relu(self.conv2(h)))            # (B, 64, 7, 7)
        h = self.pool(F.relu(self.conv3(h)))            # (B, 128, 3, 3)

        # Flatten and classify
        h = h.view(B, -1)                               # (B, 128*3*3)
        h = F.relu(self.fc1(h))                         # (B, 128)
        logits = self.fc2(h)                            # (B, num_classes)
        return logits


_cls_params = sum(p.numel() for p in NoisyClassifier().parameters())
print(f"NoisyClassifier defined — {_cls_params:,} parameters")

In [ ]:
# Train the noisy-image classifier
torch.manual_seed(42)

classifier = NoisyClassifier(num_classes=10).to(device)
cls_optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

cls_dataloader = get_mnist_dataloader(batch_size=128)
cls_data_iter = iter(cls_dataloader)
cls_losses = []

classifier.train()
for step in tqdm(range(2000), desc="Training classifier"):
    try:
        images, labels = next(cls_data_iter)
    except StopIteration:
        cls_data_iter = iter(cls_dataloader)
        images, labels = next(cls_data_iter)

    images, labels = images.to(device), labels.to(device)
    B = images.shape[0]

    # Add random noise (random timestep per sample)
    t = torch.randint(0, T, (B,), device=device)
    noise = torch.randn_like(images)
    x_t = (
        sqrt_alphas_cumprod[t, None, None, None] * images
        + sqrt_one_minus_alphas_cumprod[t, None, None, None] * noise
    )

    logits = classifier(x_t, t)
    loss = F.cross_entropy(logits, labels)

    cls_optimizer.zero_grad()
    loss.backward()
    cls_optimizer.step()
    cls_losses.append(loss.item())

print(f"Final classifier loss: {cls_losses[-1]:.4f}")
plot_loss_curve(cls_losses, title="Noisy-Image Classifier Training Loss")

### Classifier-guided sampling function

Now let's put the algorithm into code. This function runs the full DDPM reverse process, but at each step it computes the classifier gradient and shifts the noise prediction toward the target class.

In [ ]:
def classifier_guided_sample(
    model: UNet,
    classifier: NoisyClassifier,
    schedule: Dict[str, torch.Tensor],
    target_class: int,
    guidance_scale: float = 5.0,
    num_samples: int = 8,
    T: int = 1000,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    """Generate samples using classifier guidance.

    Args:
        model: Trained diffusion model (used with null class token for unconditional prediction).
        classifier: Trained noisy-image classifier.
        schedule: Noise schedule dictionary.
        target_class: Class to guide toward.
        guidance_scale: Strength of classifier guidance (s).
        num_samples: Number of images to generate.
        T: Number of diffusion timesteps.
        device: Device.

    Returns:
        (num_samples, 1, 28, 28) generated images.
    """
    model.eval()
    classifier.eval()

    null_label = 10  # null class token for 10-class model
    x = torch.randn(num_samples, 1, 28, 28, device=device)  # (B, 1, 28, 28)

    for t_idx in reversed(range(T)):
        t_batch = torch.full((num_samples,), t_idx, device=device, dtype=torch.long)

        # Step 1: unconditional noise prediction (using null token)
        with torch.no_grad():
            null_labels = torch.full((num_samples,), null_label, device=device, dtype=torch.long)
            eps_pred = model(x, t_batch, class_label=null_labels)  # (B, 1, 28, 28)

        # Step 2: classifier gradient
        x_in = x.detach().requires_grad_(True)
        logits = classifier(x_in, t_batch)
        log_probs = F.log_softmax(logits, dim=-1)
        target = torch.full((num_samples,), target_class, device=device, dtype=torch.long)
        selected_log_prob = log_probs[range(num_samples), target].sum()
        grad = torch.autograd.grad(selected_log_prob, x_in)[0]  # (B, 1, 28, 28)

        # Step 3: shift noise prediction
        alpha_bar_t = schedule["alphas_cumprod"][t_idx].to(device)
        eps_guided = eps_pred - guidance_scale * (1.0 - alpha_bar_t).sqrt() * grad

        # Step 4: DDPM sampling step
        x = ddpm_sample_step(x, eps_guided, t_idx, schedule)

    return x

### Exercise 8.2: Sweep classifier guidance scale

Let's see how guidance scale affects the outputs. Generate digit 3 with guidance scales `s = [0, 1, 2, 5, 7, 10]` and display them side by side.

- Use `classifier_guided_sample()` defined above
- Use `show_images()` to display each set of samples
- What happens at very high guidance scales? You should see moderate scales sharpen the digit, but extreme scales cause artifacts

In [ ]:
# YOUR CODE HERE — Exercise 8.2

scales = [0, 1, 2, 5, 7, 10]
target_digit = 3

# ===================== YOUR CODE HERE =====================
# For each scale in `scales`:
#   1. Set torch.manual_seed(42) for reproducibility
#   2. Call classifier_guided_sample() with the current scale
#   3. Display the result using show_images() with a title showing the scale
pass  # Replace with your implementation
# ====================== END YOUR CODE ======================

print("Classifier guidance sweep ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

scales = [0, 1, 2, 5, 7, 10]
target_digit = 3

for s in scales:
    torch.manual_seed(42)
    samples = classifier_guided_sample(
        model, classifier, schedule_cpu,
        target_class=target_digit, guidance_scale=s,
        num_samples=4, T=T, device=device,
    )
    show_images(samples, nrow=2, title=f"Classifier Guidance s={s}, digit {target_digit}")

print("At s=0 we get unconditioned (random digit) samples.")
print("As s increases, samples look more like the target digit but become less diverse.")
print("At very high s (e.g. 10), images can become oversaturated/distorted.")

---
## 8.3 — Classifier-Free Guidance (CFG)

Here's the big idea — and this is what every modern system actually uses. CFG eliminates the need for a separate classifier by training **one model** that can do both conditional and unconditional generation.

### The key idea

During training, randomly **drop the class label** (replace it with a null token) some fraction of the time (typically 10%). This teaches the model two behaviors:
- **With label:** predict noise conditioned on the class
- **Without label (null token):** predict noise unconditionally

During sampling, run **both** passes and interpolate:

$$\tilde{\epsilon} = \epsilon_\theta(x_t, t, \varnothing) + s \cdot \big(\epsilon_\theta(x_t, t, c) - \epsilon_\theta(x_t, t, \varnothing)\big)$$

### Intuition

Think of classifier-free guidance as turning up a "creativity dial." Higher guidance scale $s$ means the model follows the class label more strictly, producing sharper but less diverse outputs. At $s = 1$ you get standard conditional generation; at $s > 1$ the conditioning signal is amplified.

### Worked example: Training with label dropout

Now let's retrain the model with **10% random label dropout** — this is the only change from Exercise 8.1. Everything else (architecture, optimizer, loss) stays exactly the same.

During training, for each sample in the batch, we randomly replace the true label with the null token (index 10) with probability 0.1. This teaches the model to predict noise both conditionally and unconditionally.

In [ ]:
torch.manual_seed(42)

# Fresh model for CFG training
cfg_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4), num_classes=10).to(device)
cfg_optimizer = torch.optim.Adam(cfg_model.parameters(), lr=2e-4)

cfg_dataloader = get_mnist_dataloader(batch_size=64)
cfg_data_iter = iter(cfg_dataloader)
cfg_losses = []

p_uncond = 0.1     # probability of dropping the class label
null_label = 10    # null token index (num_classes)
num_steps = 5000

cfg_model.train()
for step in tqdm(range(num_steps), desc="Training with CFG label dropout"):
    try:
        images, labels = next(cfg_data_iter)
    except StopIteration:
        cfg_data_iter = iter(cfg_dataloader)
        images, labels = next(cfg_data_iter)

    images = images.to(device)  # (B, 1, 28, 28)
    labels = labels.to(device)  # (B,)
    B = images.shape[0]

    # THE KEY CHANGE: randomly drop class labels
    drop_mask = torch.rand(B, device=device) < p_uncond  # (B,)
    labels = labels.clone()
    labels[drop_mask] = null_label  # replace with null token

    # Standard DDPM forward process
    t = torch.randint(0, T, (B,), device=device)
    noise = torch.randn_like(images)
    x_t = (
        sqrt_alphas_cumprod[t, None, None, None] * images
        + sqrt_one_minus_alphas_cumprod[t, None, None, None] * noise
    )

    # Forward pass with (possibly dropped) labels
    noise_pred = cfg_model(x_t, t, class_label=labels)  # (B, 1, 28, 28)
    loss = F.mse_loss(noise_pred, noise)

    cfg_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(cfg_model.parameters(), 1.0)
    cfg_optimizer.step()

    cfg_losses.append(loss.item())

print(f"Final loss: {cfg_losses[-1]:.4f}")
plot_loss_curve(cfg_losses, title="CFG Training Loss (with 10% label dropout)")

---
## 8.4 — The Guidance Scale Formula

Now that we've trained a CFG model, let's look more carefully at the formula that makes it all work — and understand exactly what each guidance scale value does.

$$\tilde{\epsilon} = \underbrace{\epsilon_\theta(x_t, t, \varnothing)}_{\text{unconditional}} + s \cdot \big(\underbrace{\epsilon_\theta(x_t, t, c)}_{\text{conditional}} - \underbrace{\epsilon_\theta(x_t, t, \varnothing)}_{\text{unconditional}}\big)$$

**What the scale $s$ controls:**

| Scale | Behavior | Effect |
|-------|----------|--------|
| $s = 0$ | Pure unconditional | Ignores the class label entirely |
| $s = 1$ | Standard conditional | Normal class-conditional output |
| $s > 1$ | Amplified conditioning | Sharper, more "prototypical" outputs |
| $s \gg 1$ | Over-guided | Saturated / distorted artifacts |

Typical values: $s \in [2, 4]$ for class-conditional models, $s \in [7.5, 15]$ for text-to-image models.

### Why this works mathematically

CFG implicitly samples from a distribution proportional to:

$$\tilde{p}(x \mid c) \propto p(x) \cdot p(c \mid x)^s$$

At $s = 1$ this is the true conditional $p(x \mid c)$. At $s > 1$, the conditioning signal is **amplified** -- the model generates images that are more "prototypical" of class $c$.

### Efficient implementation

CFG requires **two forward passes** per sampling step (one conditional, one unconditional). In practice, these are batched together:

```python
# Batch both passes into a single forward call
x_in = torch.cat([x_t, x_t], dim=0)           # (2B, C, H, W)
t_in = torch.cat([t, t], dim=0)                 # (2B,)
c_in = torch.cat([class_labels, null_labels])    # (2B,)
eps_both = model(x_in, t_in, class_label=c_in)   # (2B, C, H, W)
eps_cond, eps_uncond = eps_both.chunk(2)          # each (B, C, H, W)
eps_guided = eps_uncond + scale * (eps_cond - eps_uncond)
```

### CFG sampling function

Here's the complete CFG sampling loop. It follows the same DDPM reverse process we implemented in Module 7, but at each step it:
1. Runs a **batched forward pass** with both conditional and unconditional inputs
2. Applies the CFG formula to get the guided noise prediction
3. Uses the guided prediction in the standard DDPM update

In [ ]:
@torch.no_grad()
def cfg_sample(
    model: UNet,
    schedule: Dict[str, torch.Tensor],
    class_labels: torch.Tensor,
    guidance_scale: float = 3.0,
    T: int = 1000,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    """Generate samples using classifier-free guidance.

    Args:
        model: UNet trained with label dropout (CFG-ready).
        schedule: Noise schedule dictionary.
        class_labels: (B,) desired class labels for each sample.
        guidance_scale: CFG scale (s). s=1 is standard conditional, s>1 is amplified.
        T: Number of diffusion timesteps.
        device: Device.

    Returns:
        (B, 1, 28, 28) generated images.
    """
    model.eval()
    B = class_labels.shape[0]
    null_label = model.num_classes  # null token index

    x = torch.randn(B, 1, 28, 28, device=device)  # (B, 1, 28, 28)

    for t_idx in reversed(range(T)):
        t_batch = torch.full((B,), t_idx, device=device, dtype=torch.long)  # (B,)

        # Batch conditional and unconditional forward passes
        x_in = torch.cat([x, x], dim=0)                                    # (2B, 1, 28, 28)
        t_in = torch.cat([t_batch, t_batch], dim=0)                        # (2B,)
        null_labels = torch.full((B,), null_label, device=device, dtype=torch.long)
        c_in = torch.cat([class_labels, null_labels], dim=0)               # (2B,)

        eps_both = model(x_in, t_in, class_label=c_in)                     # (2B, 1, 28, 28)
        eps_cond, eps_uncond = eps_both.chunk(2, dim=0)                     # each (B, 1, 28, 28)

        # CFG formula
        eps_guided = eps_uncond + guidance_scale * (eps_cond - eps_uncond)   # (B, 1, 28, 28)

        # DDPM reverse step
        x = ddpm_sample_step(x, eps_guided, t_idx, schedule)

    return x

In [ ]:
# Generate digit 7 at different guidance scales
target = 7
scales_to_show = [1.0, 2.0, 4.0, 8.0]

for s in scales_to_show:
    torch.manual_seed(42)
    labels_gen = torch.full((4,), target, device=device, dtype=torch.long)
    samples = cfg_sample(cfg_model, schedule_cpu, labels_gen, guidance_scale=s, T=T, device=device)
    show_images(samples, nrow=2, title=f"CFG digit {target}, s={s}")

print("Notice how higher guidance scales produce sharper, more consistent digits.")

### Exercise 8.3: CFG sampling with configurable scale

Now try it yourself. Use the `cfg_sample()` function we just built.

- Generate **8 samples** of digit 5 at guidance scales `s = [0, 0.5, 1, 2, 4, 8]`
- Display each set using `show_images()`
- Verify that `s=1` matches pure conditional generation (no guidance amplification)
- What happens at `s=0`? At `s=8`?

In [ ]:
# YOUR CODE HERE — Exercise 8.3

scales = [0, 0.5, 1, 2, 4, 8]
target_digit = 5
num_samples = 8

# ===================== YOUR CODE HERE =====================
# For each scale:
#   1. Set torch.manual_seed(42)
#   2. Create labels: torch.full((num_samples,), target_digit, device=device, dtype=torch.long)
#   3. Call cfg_sample(cfg_model, schedule_cpu, labels, guidance_scale=s, T=T, device=device)
#   4. Display with show_images(samples, nrow=4, title=f"CFG s={s}")
pass  # Replace with your implementation
# ====================== END YOUR CODE ======================

print("CFG scale sweep ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

scales = [0, 0.5, 1, 2, 4, 8]
target_digit = 5
num_samples = 8

for s in scales:
    torch.manual_seed(42)
    labels_gen = torch.full((num_samples,), target_digit, device=device, dtype=torch.long)
    samples = cfg_sample(cfg_model, schedule_cpu, labels_gen, guidance_scale=s, T=T, device=device)
    show_images(samples, nrow=4, title=f"CFG digit {target_digit}, s={s}")

print("At s=0, the model ignores the label — you get random digits.")
print("At s=1, standard conditional generation.")
print("At s=4+, sharper but less diverse — the diversity-fidelity tradeoff in action.")

---
## 8.5 — Why CFG Works: Trading Diversity for Fidelity

CFG exposes a fundamental tradeoff that shows up everywhere in generative modeling.

### The diversity-fidelity spectrum

- **$s = 1$:** standard conditional — **diverse** outputs, but some may be ambiguous
- **$s > 1$:** amplified conditioning — **higher fidelity** (images clearly match the class), but reduced diversity
- **Very high $s$:** over-guided — samples become saturated or distorted

This is analogous to **temperature** in language models. Lower temperature (high guidance) produces more predictable outputs; higher temperature (low guidance) produces more varied but less reliable outputs.

### Quantifying the tradeoff

We can measure this by generating many samples and computing:
- **Diversity:** average pairwise distance between samples (higher = more diverse)
- **Consistency:** how strongly samples match the target class (proxy for fidelity)

Let's build a simple diversity metric and sweep across guidance scales to see this in action.

In [ ]:
def compute_diversity(samples: torch.Tensor) -> float:
    """Compute average pairwise L2 distance between samples.
    
    Args:
        samples: (B, C, H, W) tensor of generated images.
    Returns:
        Mean pairwise distance (scalar).
    """
    B = samples.shape[0]
    flat = samples.reshape(B, -1)  # (B, C*H*W)
    # Pairwise L2 distances
    dists = torch.cdist(flat, flat, p=2)  # (B, B)
    # Take upper triangle (exclude self-comparisons)
    mask = torch.triu(torch.ones(B, B, dtype=torch.bool), diagonal=1)
    return dists[mask].mean().item()

print("compute_diversity() defined")


In [ ]:
# Measure diversity at different guidance scales
scales_sweep = [0.0, 0.5, 1.0, 2.0, 4.0, 6.0, 8.0]
diversities = []
num_diversity_samples = 16  # keep small for CPU feasibility

for s in tqdm(scales_sweep, desc="Diversity sweep"):
    torch.manual_seed(123)
    labels_gen = torch.full((num_diversity_samples,), 7, device=device, dtype=torch.long)
    samples = cfg_sample(
        cfg_model, schedule_cpu, labels_gen,
        guidance_scale=s, T=T, device=device,
    )
    diversities.append(compute_diversity(samples.cpu()))

# Plot
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(scales_sweep, diversities, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.set_xlabel("Guidance Scale (s)", fontsize=12)
ax.set_ylabel("Diversity (avg pairwise L2)", fontsize=12)
ax.set_title("Diversity vs. Guidance Scale", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("As guidance scale increases, diversity decreases -- samples converge toward the mode.")

---
## 8.6 — Text Conditioning Preview

Real-world systems like Stable Diffusion and DALL-E condition on **text prompts** rather than class labels. The core mechanism is **cross-attention**:

- **Q** (query) comes from the image feature maps
- **K** (key) and **V** (value) come from the text embeddings
- Each spatial location in the image attends to relevant words in the prompt

| Component | Class Conditioning | Text Conditioning |
|-----------|-------------------|-------------------|
| Input | Integer class label | Token sequence |
| Embedding | `nn.Embedding(num_classes, dim)` | Pretrained text encoder (CLIP/T5) |
| Injection | Addition to time embedding | Cross-attention (Q from image, K/V from text) |
| Expressiveness | 1 of N classes | Free-form natural language |
| CFG dropout | Replace label with null token | Replace text with empty string "" |

### CFG with text

The mechanism is identical to what we implemented with class labels. During training, the text prompt is randomly replaced with an empty string. During sampling:

$$\tilde{\epsilon} = \epsilon_\theta(x_t, t, \varnothing) + s \cdot \big(\epsilon_\theta(x_t, t, \text{text}) - \epsilon_\theta(x_t, t, \varnothing)\big)$$

We will not implement text conditioning from scratch here (it requires a pretrained text encoder), but the conceptual framework is exactly what we built in Sections 8.3--8.4 -- just with richer embeddings and cross-attention instead of addition.

---
## 8.7 — Negative Prompts: How They Work Mechanically

Now here's something fun. Recall the standard CFG formula:

$$\tilde{\epsilon} = \epsilon_{\text{uncond}} + s \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$$

### Negative prompt modification

What if we replace $\epsilon_{\text{uncond}}$ with $\epsilon_{\text{neg}}$ — the noise prediction conditioned on something we *don't* want?

$$\tilde{\epsilon} = \epsilon_{\text{neg}} + s \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{neg}})$$

This steers the generation **away from** the negative prompt and **toward** the positive prompt. The larger $s$ is, the stronger both effects.

### How it works in practice

**Example in text-to-image:**
- Positive prompt: "a photograph of a cat"
- Negative prompt: "blurry, low quality"
- Result: the model generates a cat while actively avoiding blurriness

### With class labels

We can demonstrate the same idea with class labels: use one class as the positive target and another as the negative. The model generates images that look like the positive class while avoiding the negative class.

In [ ]:
@torch.no_grad()
def cfg_sample_with_negative(
    model: UNet,
    schedule: Dict[str, torch.Tensor],
    class_labels: torch.Tensor,
    negative_labels: torch.Tensor,
    guidance_scale: float = 3.0,
    T: int = 1000,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    """CFG sampling with negative prompt (class) support.

    Instead of using the unconditional prediction as the baseline,
    uses the negative class prediction. This steers away from the
    negative class and toward the positive class.

    Args:
        model: CFG-trained UNet.
        schedule: Noise schedule dictionary.
        class_labels: (B,) positive class labels.
        negative_labels: (B,) negative class labels (steer away from these).
        guidance_scale: CFG scale.
        T: Timesteps.
        device: Device.

    Returns:
        (B, 1, 28, 28) generated images.
    """
    model.eval()
    B = class_labels.shape[0]

    x = torch.randn(B, 1, 28, 28, device=device)

    for t_idx in reversed(range(T)):
        t_batch = torch.full((B,), t_idx, device=device, dtype=torch.long)

        # Batch positive and negative forward passes
        x_in = torch.cat([x, x], dim=0)                              # (2B, 1, 28, 28)
        t_in = torch.cat([t_batch, t_batch], dim=0)                  # (2B,)
        c_in = torch.cat([class_labels, negative_labels], dim=0)     # (2B,)

        eps_both = model(x_in, t_in, class_label=c_in)               # (2B, 1, 28, 28)
        eps_cond, eps_neg = eps_both.chunk(2, dim=0)                  # each (B, 1, 28, 28)

        # Negative prompt CFG formula
        eps_guided = eps_neg + guidance_scale * (eps_cond - eps_neg)  # (B, 1, 28, 28)

        x = ddpm_sample_step(x, eps_guided, t_idx, schedule)

    return x

### Exercise 8.4: Negative prompts with class labels

Try this yourself. Generate digit 1 with digit 7 as the negative prompt, then compare against standard CFG (where the negative is the null token).

- Use `cfg_sample()` for standard CFG and `cfg_sample_with_negative()` for negative prompts
- Use guidance scale `s=4.0` and generate 8 samples each
- Display both sets with `show_images()` — you should see that negative prompts steer the output away from the negative class

In [ ]:
# YOUR CODE HERE — Exercise 8.4

B = 8
pos_labels = torch.full((B,), 1, device=device, dtype=torch.long)

# ===================== YOUR CODE HERE =====================
# 1. Generate digit 1 with standard CFG (use cfg_sample with guidance_scale=4.0)
#    Set torch.manual_seed(42) before sampling
# samples_standard = ...

# 2. Generate digit 1 with digit 7 as negative prompt (use cfg_sample_with_negative)
#    Set torch.manual_seed(42) before sampling
#    neg_labels = torch.full((B,), 7, device=device, dtype=torch.long)
# samples_neg = ...

# 3. Display both with show_images()
# show_images(samples_standard, nrow=4, title="Standard CFG (null negative)")
# show_images(samples_neg, nrow=4, title="Negative prompt: digit 7")
pass  # Replace with your implementation
# ====================== END YOUR CODE ======================

print("Negative prompt exercise ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

B = 8
pos_labels = torch.full((B,), 1, device=device, dtype=torch.long)
neg_7_labels = torch.full((B,), 7, device=device, dtype=torch.long)

# Standard CFG (null as negative)
torch.manual_seed(42)
samples_standard = cfg_sample(
    cfg_model, schedule_cpu, pos_labels,
    guidance_scale=4.0, T=T, device=device,
)

# Negative prompt: steer away from 7
torch.manual_seed(42)
samples_neg = cfg_sample_with_negative(
    cfg_model, schedule_cpu, pos_labels, neg_7_labels,
    guidance_scale=4.0, T=T, device=device,
)

show_images(samples_standard, nrow=4, title="Standard CFG (null negative)")
show_images(samples_neg, nrow=4, title="Negative prompt: digit 7")

print("With the negative prompt, the model steers away from digit-7-like features")
print("(e.g., avoiding diagonal strokes that 7s tend to have).")

---
## Capstone Exercise: Full Class-Conditional MNIST Diffusion with CFG

Let's bring everything together in a complete pipeline. This is the culmination of the module — by the end, you'll have a model that generates any MNIST digit on demand.

1. Build a `UNet` with `num_classes=10` (null token at index 10)
2. Train with 10% random label dropout for 8000 steps
3. Implement CFG-guided sampling
4. Generate a grid: **rows** = digits 0–9, **columns** = guidance scales 1, 2, 4, 8
5. Generate "the digit 7" at different scales
6. Compare unconditional ($s=0$) vs. guided ($s=4$)

In [ ]:
# YOUR CODE HERE — Capstone Exercise
#
# Part 1: Train a CFG model (8000 steps with 10% label dropout)
# Part 2: Generate digits 0-9 grid at scales [1, 2, 4, 8]
# Part 3: Generate digit 7 at scales [0, 1, 2, 4, 8]
# Part 4: Compare unconditional (s=0) vs guided (s=4)
#
# You can reuse UNet, cfg_sample, cosine_schedule, and other utilities.
# The CFG training loop from the worked example above is a good starting point.

# ===================== YOUR CODE HERE =====================

# --- Part 1: Training ---
# cap_model = UNet(image_channels=1, base_channels=64, channel_mults=(1,2,4), num_classes=10).to(device)
# Train for 8000 steps with 10% label dropout (p_uncond=0.1)

# --- Part 2: Digit x Scale Grid ---
# For each digit 0-9 and each scale in [1, 2, 4, 8]:
#   generate 1 sample and display in a grid

# --- Part 3: Digit 7 sweep ---
# Generate 8 samples of digit 7 at each scale [0, 1, 2, 4, 8]

# --- Part 4: Unconditional vs Guided ---
# Generate 16 samples at s=0 (unconditional) and s=4 (guided)
# Display side by side with show_images()

pass  # Replace with your implementation
# ====================== END YOUR CODE ======================

In [ ]:
# ✅ SOLUTION — try the exercise above before running this — Part 1: Training
torch.manual_seed(42)

# Setup
T_cap = 1000
schedule_cap = cosine_schedule(T_cap)
schedule_cap_device = prepare_schedule(schedule_cap, device)

cap_dataloader = get_mnist_dataloader(batch_size=64)
cap_model = UNet(
    image_channels=1,
    base_channels=64,
    channel_mults=(1, 2, 4),
    num_classes=10,
).to(device)
cap_optimizer = torch.optim.Adam(cap_model.parameters(), lr=2e-4)

p_uncond_cap = 0.1  # 10% label dropout
null_label_cap = 10
num_steps_cap = 8000
cap_losses = []
cap_data_iter = iter(cap_dataloader)

cap_model.train()
for step in tqdm(range(num_steps_cap), desc="Capstone Training"):
    try:
        images, labels = next(cap_data_iter)
    except StopIteration:
        cap_data_iter = iter(cap_dataloader)
        images, labels = next(cap_data_iter)

    images = images.to(device)  # (B, 1, 28, 28)
    labels = labels.to(device)  # (B,)
    B = images.shape[0]

    # Random label dropout for CFG
    drop_mask = torch.rand(B, device=device) < p_uncond_cap
    labels = labels.clone()
    labels[drop_mask] = null_label_cap

    # Forward process
    t = torch.randint(0, T_cap, (B,), device=device)
    noise = torch.randn_like(images)
    x_t = (
        schedule_cap_device["sqrt_alphas_cumprod"][t, None, None, None] * images
        + schedule_cap_device["sqrt_one_minus_alphas_cumprod"][t, None, None, None] * noise
    )

    # Noise prediction and loss
    noise_pred = cap_model(x_t, t, class_label=labels)
    loss = F.mse_loss(noise_pred, noise)

    cap_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(cap_model.parameters(), 1.0)
    cap_optimizer.step()

    cap_losses.append(loss.item())

print(f"Final loss: {cap_losses[-1]:.4f}")
plot_loss_curve(cap_losses, title="Capstone: CFG Training Loss (8000 steps)")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this — Part 2: Grid of digits x guidance scales
# Rows = digits 0-9, Columns = guidance scales 1, 2, 4, 8

cap_scales = [1.0, 2.0, 4.0, 8.0]
num_digits = 10

fig, axes = plt.subplots(num_digits, len(cap_scales), figsize=(3 * len(cap_scales), 2 * num_digits))

for row, digit in enumerate(range(num_digits)):
    for col, s in enumerate(cap_scales):
        torch.manual_seed(42)
        labels_gen = torch.full((1,), digit, device=device, dtype=torch.long)
        sample = cfg_sample(
            cap_model, schedule_cap, labels_gen,
            guidance_scale=s, T=T_cap, device=device,
        )
        img = denormalize(sample[0].cpu()).squeeze(0).numpy()  # (28, 28)
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(f"s = {s}", fontsize=12)
    axes[row, 0].set_ylabel(f"Digit {digit}", fontsize=11, rotation=0, labelpad=40)

plt.suptitle("Capstone: Digits 0-9 at Different CFG Scales", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ SOLUTION — Capstone Part 3: "Generate the digit 7" at different scales

scales_7 = [0.0, 1.0, 2.0, 4.0, 8.0]

for s in scales_7:
    torch.manual_seed(42)
    labels_gen = torch.full((8,), 7, device=device, dtype=torch.long)
    samples = cfg_sample(
        cap_model, schedule_cap, labels_gen,
        guidance_scale=s, T=T_cap, device=device,
    )
    show_images(samples, nrow=4, title=f"Digit 7, CFG s={s}")

print("Watch how the digits sharpen as guidance scale increases!")

In [ ]:
# ✅ SOLUTION — Capstone Part 4: Unconditional (s=0) vs Guided (s=4)

# Unconditional: s=0 (class label is irrelevant since guidance is zero)
torch.manual_seed(42)
labels_gen = torch.full((16,), 7, device=device, dtype=torch.long)
samples_uncond = cfg_sample(
    cap_model, schedule_cap, labels_gen,
    guidance_scale=0.0, T=T_cap, device=device,
)
show_images(samples_uncond, nrow=8, title="Unconditional (s=0)")

# Guided: s=4
torch.manual_seed(42)
samples_guided = cfg_sample(
    cap_model, schedule_cap, labels_gen,
    guidance_scale=4.0, T=T_cap, device=device,
)
show_images(samples_guided, nrow=8, title="Guided: digit 7, s=4")

print("At s=0, the model ignores the class label and generates random digits.")
print("At s=4, the model consistently generates the target digit with high fidelity.")
print("\nCongratulations! You've built a complete class-conditional diffusion model with CFG.")
print("This is the same mechanism powering Stable Diffusion, DALL-E, and Midjourney.")

---
## Summary

Here's everything we covered — and you now have working implementations of all of it:

| Concept | Key Idea |
|---------|----------|
| **Class conditioning** | Add class embedding to timestep embedding; injection via addition to the existing MLP path |
| **Classifier guidance** | Use external classifier gradient to steer sampling (requires separate noisy-image classifier) |
| **Classifier-free guidance** | Drop labels randomly during training; one model does both conditional and unconditional |
| **Guidance scale** | $\tilde{\epsilon} = \epsilon_{\text{uncond}} + s(\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$; $s=1$ is standard, $s>1$ trades diversity for fidelity |
| **Text conditioning** | Cross-attention with text encoder embeddings; same CFG framework, richer conditioning signal |
| **Negative prompts** | Replace $\epsilon_{\text{uncond}}$ with $\epsilon_{\text{neg}}$ in CFG formula; steers away from undesired features |

**Key takeaways:**
- CFG is used in every major text-to-image system (Stable Diffusion, DALL-E 3, Imagen, Midjourney)
- The training change is minimal: randomly drop labels with ~10% probability
- The sampling change requires two forward passes per step (or one batched pass)
- Guidance scale is one of the most impactful hyperparameters for output quality
- Mathematically, CFG samples from $p(x) \cdot p(c|x)^s$, amplifying the conditional signal

In the next module, we'll tackle **latent diffusion** — moving from pixel space to a compressed latent space for dramatically faster generation.